In [ ]:
!nvidia-smi
%matplotlib inline

In [1]:
import os
import pandas as pd
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from tqdm import tqdm
from typing import List, Literal, List, Dict, Any, Optional
import numpy as np
import seaborn as sns

from datasets import load_dataset
import random
import json
import re
from functools import partial
from datasets import Dataset
from copy import deepcopy
import evaluate
import nltk
from scipy.stats import ttest_ind
import string
from collections import Counter

import openai
import os
import time
import pandas as pd
import torch

from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from dotenv import load_dotenv
load_dotenv()

/home/yhuang/ondemand/paper_clean/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## QA implementations

### Setup

In [2]:
underspecified_set = load_dataset(
    "json",
data_files="./intermediate/BASELINE_classified_GaRAGe_sample_UND.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

fully_specified_set = load_dataset(
    "json",
data_files="./intermediate/BASELINE_classified_GaRAGe_sample_FS.jsonl",
    split="all"
)

### Implementation

In [3]:
from openai import OpenAI
google_api = os.environ.get("GOOGLE_API_KEY")
client = OpenAI(api_key=google_api, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [4]:
from helper_functions_qa import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)

In [5]:
df_UND = pd.read_json('./intermediate/BASELINE_classified_GaRAGe_sample_UND.jsonl', lines=True)
df_FS = pd.read_json('./intermediate/BASELINE_classified_GaRAGe_sample_FS.jsonl', lines=True)

In [6]:
df_UND

,id,question,answer,qwen3_thinking,qwen3_model_response,qwen3_model_pred
0,a881c11a-4ed7-4a9a-a4ee-87afcc427b65,What strategies can businesses employ to mitig...,"[To mitigate ASC 842 compliance challenges, bu...","<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What strategies can businesses ...",underspecified
1,38512574-e68d-40fd-ae97-2162db411df7,How does the Deep Retinal Convolution Neural N...,[The Deep Retinal Convolutional Neural Network...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""How does the Deep Retinal Con...",underspecified
2,709d11b3-eb14-43be-9a17-d567eb7abd40,"How does Zoe Law's ""Legends"" exhibition reflec...","[The ""Legends"" exhibition by Zoë Law reflects ...","<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How does Zoe Law's \""Legends\"" ...",underspecified
3,19805270-2046-4431-9561-45d95e8b317b,How does the involvement of Tyco Ventures and ...,[The involvement of Tyco Ventures and Integral...,"<think>\nOkay, let's see. The user is asking h...","{\n ""query"": ""How does the involvement of Tyc...",underspecified
4,e59f3a34-a831-44a0-a379-dcb99fc4305c,How has the Drake-Kendrick Lamar feud influenc...,[The feud between Drake and Kendrick Lamar has...,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""How has the Drake-Kendrick La...",underspecified
...,...,...,...,...,...,...
598,a57d6b79-4f64-42ac-a304-1ad2244a194d,How did Elon Musk's opposition influence the g...,[Elon Musk's opposition led to the president-e...,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How did Elon Musk's opposition ...",underspecified
599,154d2c6c-4fbb-49f1-8241-61ffc0b24d8b,What challenges does self-managed OpenSearch d...,[Self-managed OpenSearch deployments face seve...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What challenges does self-manag...",underspecified
600,e39aaa94-ce43-4b0a-8701-99d737a0d066,What are the implications of Cleveland-Cliffs ...,[Cleveland-Cliffs CEO's plan to make another o...,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""What are the implications of Cl...",underspecified
601,1ca2aa68-7031-49b3-98d9-4cb52e06571c,How has the expansion of telehealth services u...,[The expansion of telehealth services under Me...,"<think>\nOkay, let's see. The user asked how t...","{\n ""query"": ""How has the expansion of telehe...",underspecified


In [7]:
df_FS

,id,question,answer,qwen3_thinking,qwen3_model_response,qwen3_model_pred
0,76cc2729-2ec6-4100-b9c1-59c0a1b3d3e9,How does Amazon Cognito support passwordless a...,[Amazon Cognito supports passwordless authenti...,"<think>\nOkay, let's see. The user is asking h...","{\n ""query"": ""How does Amazon Cognito support...",fully specified
1,ae94948d-eff9-4c5c-aa5e-b0f8131c7cc4,How does the integration of Radiata's technolo...,[The integration of Radiata's technology into ...,"<think>\nOkay, let me try to figure this out. ...","{\n ""query"": ""How does the integration of Rad...",fully specified
2,178c2659-4512-4fb7-882f-a47d7a756dcd,How do TMD factorization and collinear factori...,[TMD factorization and collinear factorization...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""How do TMD factorization and co...",fully specified
3,5551da4f-3468-48fe-895e-57213d573a2a,What advanced routing policies does Amazon Rou...,[Amazon Route 53 supports several advanced rou...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What advanced routing policies ...",fully specified
4,00140d51-5e44-417c-b14d-5102d1ce3e42,What are the key differences between Gemini 1....,[Gemini 1.5 Pro utilizes a transformer-based a...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What are the key differences be...",fully specified
...,...,...,...,...,...,...
392,7ad63dc9-235a-4fb4-9196-74552c0d6e3f,What implications do Meta's updated content po...,"[Meta's updated content policies, which replac...","<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""What implications do Meta's upd...",fully specified
393,5c372969-523f-42dd-893a-6fa33e1b1b0a,How did Donald Trump's campaign respond to the...,[Donald Trump's campaign responded to the assa...,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How did Donald Trump's campaign...",fully specified
394,775d97fd-4033-4373-898e-9c222e7f180a,What are the top reader interests and policy p...,[There is not enough grounding for an answer.],"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""What are the top reader interes...",fully specified
395,1d666b62-055e-46ea-abc7-9ea223e753f8,How does Abbott's Confirm Rx Insertable Cardia...,[Abbott's Confirm Rx Insertable Cardiac Monito...,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How does Abbott's Confirm Rx In...",fully specified


In [8]:
short_results_UND = batch_QA_with_progress(
    underspecified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_short_answer",
    fill_value=["error"],
    client=client,
    model="gemini-2.5-flash",
    temperature=0.0
)

Running model_short_answer: 100%|██████████| 61/61 [38:22<00:00, 37.74s/it]


In [9]:
# batch QA for FS
short_results_FS = batch_QA_with_progress(
    fully_specified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_short_answer",
    fill_value=["error"],
    client=client,
    model="gemini-2.5-flash",
    temperature=0.0
)

Running model_short_answer: 100%|██████████| 40/40 [23:16<00:00, 34.90s/it]


In [10]:
qa_underspecified = deepcopy(underspecified_set)

for key in short_results_UND:
    qa_underspecified = qa_underspecified.add_column(key, short_results_UND[key])

qa_underspecified.to_json("./intermediate/BASELINE_GaRAGe_UND_qa_Gemini.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 11.15ba/s]


2791219

In [11]:
qa_fully_specified = deepcopy(fully_specified_set)

for key in short_results_FS:
    qa_fully_specified = qa_fully_specified.add_column(key, short_results_FS[key])

qa_fully_specified.to_json("./intermediate/BASELINE_GaRAGe_FS_qa_Gemini.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 90.05ba/s]


1868982

In [12]:
df = pd.read_json("./intermediate/BASELINE_GaRAGe_UND_qa_Gemini.jsonl", lines=True)
df.to_csv('./intermediate/BASELINE_GaRAGe_UND_qa_Gemini.csv')

df = pd.read_json("./intermediate/BASELINE_GaRAGe_FS_qa_Gemini.jsonl", lines=True)
df.to_csv('./intermediate/BASELINE_GaRAGe_FS_qa_Gemini.csv')

## Evaluations

In [13]:
underspecified_set_qa = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_GaRAGe_UND_qa_Gemini.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

fully_specified_set_qa = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_GaRAGe_FS_qa_Gemini.jsonl",
    split="all"
)

Generating train split: 603 examples [00:00, 28935.83 examples/s]
Generating train split: 397 examples [00:00, 71452.91 examples/s]


### Squad EM + F1

In [14]:
from helper_functions_qa import evaluate_squad_per_sample_multi_ref_pred

In [15]:
# Official squad script for avg EM and F1, not possible for t-test


# Evaluate fully specified subset
dataset = load_dataset("json", data_files="./intermediate/BASELINE_GaRAGe_FS_qa_Gemini.jsonl", split="all")

# 加载 HuggingFace 的 squad 评估器
squad_metric = evaluate.load("squad")

# 构造 predictions 和 references（标准格式）
predictions = [
    {
        "id": str(i),
        "prediction_text": pred[0] if isinstance(pred, list) and pred else ""
    }
    for i, pred in enumerate(dataset["model_short_answer"])
]

references = [
    {
        "id": str(i),
        "answers": {
            "text": ref if isinstance(ref, list) else [ref],
            "answer_start": [0] * len(ref if isinstance(ref, list) else [ref])
        }
    }
    for i, ref in enumerate(dataset["answer"])
]

# 计算 SQuAD-style EM 和 F1
results = squad_metric.compute(predictions=predictions, references=references)

# 打印平均指标
print(f"Exact Match: {results['exact_match']:.2f}")
print(f"F1 Score: {results['f1']:.2f}")

Exact Match: 0.00
F1 Score: 17.10


In [16]:
# Official squad script for avg EM and F1, not possible for t-test


# Evaluate fully specified subset
dataset = load_dataset("json", data_files="./intermediate/BASELINE_GaRAGe_UND_qa_Gemini.jsonl", split="all")

# 加载 HuggingFace 的 squad 评估器
squad_metric = evaluate.load("squad")

# 构造 predictions 和 references（标准格式）
predictions = [
    {
        "id": str(i),
        "prediction_text": pred[0] if isinstance(pred, list) and pred else ""
    }
    for i, pred in enumerate(dataset["model_short_answer"])
]

references = [
    {
        "id": str(i),
        "answers": {
            "text": ref if isinstance(ref, list) else [ref],
            "answer_start": [0] * len(ref if isinstance(ref, list) else [ref])
        }
    }
    for i, ref in enumerate(dataset["answer"])
]

# 计算 SQuAD-style EM 和 F1
results = squad_metric.compute(predictions=predictions, references=references)

# 打印平均指标
print(f"Exact Match: {results['exact_match']:.2f}")
print(f"F1 Score: {results['f1']:.2f}")

Exact Match: 0.00
F1 Score: 15.41


In [17]:
squad_scored_UND, UND_f1_list, UND_em_list = evaluate_squad_per_sample_multi_ref_pred(underspecified_set_qa, ref_col="answer")
squad_scored_UND.to_json("./intermediate/BASELINE_GaRAGe_UND_qa_Gemini_with_squad_scores.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 68.07ba/s]


2806842

In [18]:
squad_scored_FS, FS_f1_list, FS_em_list = evaluate_squad_per_sample_multi_ref_pred(fully_specified_set_qa, ref_col="answer")
squad_scored_FS.to_json("./intermediate/BASELINE_GaRAGe_FS_qa_Gemini_with_squad_scores.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 109.38ba/s]


1879295

In [19]:
df = pd.read_json("./intermediate/BASELINE_GaRAGe_UND_qa_Gemini_with_squad_scores.jsonl", lines=True)
df.to_csv('./intermediate/BASELINE_GaRAGe_UND_qa_Gemini_with_squad_scores.csv')

df = pd.read_json("./intermediate/BASELINE_GaRAGe_FS_qa_Gemini_with_squad_scores.jsonl", lines=True)
df.to_csv('./intermediate/BASELINE_GaRAGe_FS_qa_Gemini_with_squad_scores.csv')

In [20]:
UND_mean_em = np.mean(UND_em_list)  # em_scores: EM list per sample
UND_mean_f1 = np.mean(UND_f1_list)  # f1_scores F1 list per sample
print(f"UND Exact Match (avg): {UND_mean_em * 100:.2f}")
print(f"UND F1 Score (avg): {UND_mean_f1 * 100:.2f}")

FS_mean_em = np.mean(FS_em_list)  # em_scores: EM list per sample
FS_mean_f1 = np.mean(FS_f1_list)  # f1_scores F1 list per sample
print(f"FS Exact Match (avg): {FS_mean_em * 100:.2f}")
print(f"FS F1 Score (avg): {FS_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(FS_f1_list, UND_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(FS_em_list, UND_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

UND Exact Match (avg): 0.00
UND F1 Score (avg): 15.41
FS Exact Match (avg): 0.00
FS F1 Score (avg): 17.11
F1: t=2.521, p=0.0119
EM: t=nan, p=nan


### Ragas

In [21]:
from helper_functions_qa import answer_accuracy

In [22]:
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

UND_full = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_GaRAGe_UND_qa_Gemini_with_squad_scores.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

FS_full = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_GaRAGe_FS_qa_Gemini_with_squad_scores.jsonl",
    split="all"
)

Generating train split: 603 examples [00:00, 66590.28 examples/s]
Generating train split: 397 examples [00:00, 55813.46 examples/s]


In [23]:
UND_ragas = await answer_accuracy(UND_full, evaluator_llm, ref_col = "answer")
UND_ragas.to_csv("./output_csv/GaRAGe_UND_Gemini_Ragas.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 17.26ba/s]


2723522

In [24]:
FS_ragas = await answer_accuracy(FS_full, evaluator_llm, ref_col = "answer")
FS_ragas.to_csv("./output_csv/GaRAGe_FS_Gemini_Ragas.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 25.64ba/s]


1824615

In [25]:
UND_ragas_AA = list(UND_ragas["ragas_AA_short"])
FS_ragas_AA = list(FS_ragas["ragas_AA_short"])

UND_mean_AA = np.mean(UND_ragas_AA)
print(f"UND AA (avg): {UND_mean_AA * 100:.2f}")


FS_mean_AA = np.mean(FS_ragas_AA)
print(f"FS AA (avg): {FS_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(FS_ragas_AA, UND_ragas_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

UND AA (avg): 50.08
FS AA (avg): 57.24
AA: t=2.591, p=0.0097
